# W2 · 让板子出声 —— 音频通路自检

这一格的目标只有一个：**证明整条音频通路是活的**，并量出「样本是谁在搬」。

「对着麦克风说话、耳机里听到」这件事没法写成代码，所以把它拆成两件可复现的事：

1. 放一段数学上已知干净的正弦波 —— 出声 = 通路是活的
2. 用 CPU 时间证明样本是谁搬的 —— 这决定了自研核要替掉什么

**跑之前**：耳机插板子的 `HP + Mic` 口，音量开到听得清。

> 为什么不直接用手上那段 `recording_0.wav`：它底噪重、电平只有满量程的 25%，
> 分不清杂音是板子的还是素材的。换成自己生成的正弦波，如果还有杂音，
> 问题就一定在板子这一侧。


In [ ]:
%matplotlib inline
import math, os, resource, struct, time, wave

import numpy as np
import matplotlib.pyplot as plt
from pynq.overlays.base import BaseOverlay

# 图里的文字一律用英文。实测（2026-09-27，板子自带的字体）：
#     默认 DejaVu Sans          -> 数字/英文正常，中文变方框
#     改设 Droid Sans Fallback  -> 中文正常，数字/英文变方框
# matplotlib 3.1 不会在同一个字符串里逐字回退，两个字体只能二选一。
# 坐标轴刻度全是数字，所以选英文标签。
# （板子上带中文字形的字体只有一个：fc-list :lang=zh 查到 Droid Sans Fallback）

SAMPLE_RATE = 48000          # codec 固定 48 kHz
TONE_HZ = 440.0              # A4，听得最清楚
DURATION_S = 3.0
WAV_PATH = "/home/xilinx/w2_tone440.wav"
REC_PATH = "/home/xilinx/w2_recording.wav"
VOLUME = 62                  # 允许的最大值是 62，不是 63


def make_tone(path, hz=TONE_HZ, seconds=DURATION_S, rate=SAMPLE_RATE, amp=0.5):
    """写一个干净的 24 bit / 双声道 / 48 kHz 正弦波 wav。

    amp 取 0.5（满量程一半）：比手上那段录音（只有 25%）响得多，又不会削顶。
    """
    peak = int(amp * (2 ** 23 - 1))
    frames = bytearray()
    for i in range(int(rate * seconds)):
        v = int(peak * math.sin(2 * math.pi * hz * i / rate))
        raw = struct.pack("<i", v)[:3]       # 小端 24 bit，取低 3 字节
        frames += raw + raw                  # 左右声道放同一份
    with wave.open(path, "wb") as f:
        f.setnchannels(2)
        f.setsampwidth(3)
        f.setframerate(rate)
        f.writeframes(bytes(frames))
    return path


def read_wav(path):
    """读回 24 bit wav，还原成有符号整数。"""
    with wave.open(path, "rb") as f:
        n, width, rate = f.getnframes(), f.getsampwidth(), f.getframerate()
        raw = f.readframes(n)
    if width != 3:
        raise ValueError("本项目的 wav 都是 24 bit，读到的是 %d bit" % (width * 8))
    a = np.frombuffer(raw, dtype=np.uint8).reshape(-1, 3)
    x = (a[:, 0].astype(np.int32)
         | (a[:, 1].astype(np.int32) << 8)
         | (a[:, 2].astype(np.int32) << 16))
    x = np.where(x >= 1 << 23, x - (1 << 24), x)   # 24 位补码还原
    return x.reshape(-1, 2), rate


if not os.path.exists(WAV_PATH):
    make_tone(WAV_PATH)
print("测试音频已就绪：", WAV_PATH)


In [ ]:
# 先看一眼要放的信号长什么样 —— 这是后面所有对比的「已知输入」
x, rate = read_wav(WAV_PATH)
print("采样率 %d Hz，%d 帧（%.2f 秒），峰值 %d / %d"
      % (rate, len(x), len(x) / rate, np.abs(x).max(), (1 << 23) - 1))

t = np.arange(len(x)) / rate
plt.figure(figsize=(10, 3))
plt.plot(t[:2400], x[:2400, 0])
plt.title("w2_tone440.wav, first 50 ms -- should be a clean sine")
plt.xlabel("time (s)"); plt.ylabel("sample value"); plt.grid(True); plt.tight_layout()
plt.show()


In [ ]:
def measure(fn, *args):
    """跑一次 fn，返回 (墙钟秒数, 进程 CPU 秒数)。

    ru_utime / ru_stime 是进程「累计」值，必须先取基准再差分 ——
    直接读会把之前所有工作（比如加载 overlay）一起算进来，
    曾算出过「墙钟 2.96 s / CPU 13.21 s」这种超过双核上限的荒谬数。
    """
    r0 = resource.getrusage(resource.RUSAGE_SELF)
    t0 = time.time()
    fn(*args)
    wall = time.time() - t0
    r1 = resource.getrusage(resource.RUSAGE_SELF)
    cpu = (r1.ru_utime - r0.ru_utime) + (r1.ru_stime - r0.ru_stime)
    return wall, cpu


# 板子开机时出厂固件已经把 base overlay 灌好了，这里只是取个句柄
audio = BaseOverlay("base.bit").audio
audio.set_volume(VOLUME)
audio.load(WAV_PATH)
print("已加载 %s，音量 %d" % (WAV_PATH, VOLUME))

print("播放中（应该听到一声干净的蜂鸣）...")
play_wall, play_cpu = measure(audio.play)
print("播放   墙钟 %.2f 秒   CPU %.2f 秒" % (play_wall, play_cpu))


In [ ]:
print("录音中 ...")
rec_wall, rec_cpu = measure(audio.record, DURATION_S)
print("录音   墙钟 %.2f 秒   CPU %.2f 秒" % (rec_wall, rec_cpu))
audio.save(REC_PATH)
print("已存到", REC_PATH)

y, rate2 = read_wav(REC_PATH)
plt.figure(figsize=(10, 3))
plt.plot(np.arange(len(y))[:2400] / rate2, y[:2400, 0])
plt.title("w2_recording.wav, first 50 ms -- whatever the HP+Mic jack is fed")
plt.xlabel("time (s)"); plt.ylabel("sample value"); plt.grid(True); plt.tight_layout()
plt.show()


## 怎么读上面这两个数

| 动作 | 墙钟 | CPU |
|---|---|---|
| 播放 3 秒 | 2.96 s | **2.96 s** |
| 录音 3 秒 | 2.95 s | **2.95 s** |

这块芯片是 **2 个核**，所以「CPU ≈ 墙钟」等于 **一个核被 100% 占满**。

如果样本是 FPGA 搬的，CPU 应该几乎不花时间。它花了 100%，说明是 CPU 在一个数字一个数字地亲手搬。

旁证也对得上：出厂 overlay 里音频那块电路的接口只有一种，叫 `S_AXI`（从接口）——
这种接口只能被 CPU 读和写，自己没有去内存取数据的能力。整条音频通路里**没有任何 DMA**。

**这就是本项目存在的理由。** 要把「搬运」和「计算」一起从 CPU 手里拿走：

- 搬运 → 交给硬件的传送带（DMA）
- 计算 → 交给 FPGA 上我们自己的电路（滤波核）

CPU 只需要在开头说一句「开始」，然后去干别的。

## 顺带发现：录到的信号带直流偏置

第二张图不是围绕 0 上下摆动的，而是整体抬高了约 10 万（约满量程的 1.2%），
并且缓慢下滑 —— 这是采集通路的**直流偏置**。

以后算频谱或做前后对比之前要**先减掉均值**，否则频谱图会在 0 Hz 处冒出一根很高的假峰，
把真正要看的东西压下去。
